In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import io
from matplotlib.patches import Patch
import re


CUSTOM_BINS_LENGTH = 5
CUSTOM_BINS_MINUTES = list(range(CUSTOM_BINS_LENGTH, 30+CUSTOM_BINS_LENGTH, CUSTOM_BINS_LENGTH))

# Prefixes to look for in the dataframe
METRIC_PREFIXES = ['Di', 'Do', 'Si', 'So']

# ==========================================
# 1. LOAD DATA
# ==========================================
sample_data = """
Di_pre	Di_1min	Di_2min	Di_3min	Di_4min	Di_5min	Di_6min	Di_7min	Di_8min	Di_9min	Di_10min	Di_11min	Di_12min	Di_13min	Di_14min	Di_15min	Di_16min	Di_17min	Di_18min	Di_19min	Di_20min	Di_21min	Di_22min	Di_23min	Di_24min	Di_25min	Di_26min	Di_27min	Di_28min	Di_29min	Di_30min	Di_Sum	Do_pre	Do_1min	Do_2min	Do_3min	Do_4min	Do_5min	Do_6min	Do_7min	Do_8min	Do_9min	Do_10min	Do_11min	Do_12min	Do_13min	Do_14min	Do_15min	Do_16min	Do_17min	Do_18min	Do_19min	Do_20min	Do_21min	Do_22min	Do_23min	Do_24min	Do_25min	Do_26min	Do_27min	Do_28min	Do_29min	Do_30min	Do_Sum	Si_pre	Si_1min	Si_2min	Si_3min	Si_4min	Si_5min	Si_6min	Si_7min	Si_8min	Si_9min	Si_10min	Si_11min	Si_12min	Si_13min	Si_14min	Si_15min	Si_16min	Si_17min	Si_18min	Si_19min	Si_20min	Si_21min	Si_22min	Si_23min	Si_24min	Si_25min	Si_26min	Si_27min	Si_28min	Si_29min	Si_30min	Si_Sum	So_pre	So_1min	So_2min	So_3min	So_4min	So_5min	So_6min	So_7min	So_8min	So_9min	So_10min	So_11min	So_12min	So_13min	So_14min	So_15min	So_16min	So_17min	So_18min	So_19min	So_20min	So_21min	So_22min	So_23min	So_24min	So_25min	So_26min	So_27min	So_28min	So_29min	So_30min	So_Sum	Estrous	ID

"""

df = pd.read_csv(io.StringIO(sample_data), sep='\t')

# Ensure 'Fem' or 'No.' is used as ID if present, otherwise use index
df['Fem'] = df['ID']
df['Estrous'] = df['Estrous'].astype('category')

# ==========================================
# 2. DYNAMIC BIN CALCULATION
# ==========================================

def get_time_columns(df, prefix):
    """
    Identifies all time-point columns for a given prefix (e.g., 'Di').
    Returns a dict: { minute_int: column_name }
    Also identifies 'pre' and 'Sum' if they exist.
    """
    cols = {}
    pre_col = f"{prefix}_pre"
    sum_col = f"{prefix}_Sum"
    
    if pre_col in df.columns:
        cols['pre'] = pre_col
        
    if sum_col in df.columns:
        cols['sum'] = sum_col

    # Regex to find columns like Di_1min, Di_10min, etc.
    pattern = re.compile(rf"^{prefix}_(\d+)min$")
    
    for col in df.columns:
        match = pattern.match(col)
        if match:
            minute = int(match.group(1))
            cols[minute] = col
            
    return cols

def calculate_cumulative_bin(df, prefix, target_minute, col_map):
    """
    Calculates the cumulative sum for a specific prefix up to target_minute.
    """
    # Get all minute keys <= target_minute
    relevant_minutes = [m for m in col_map.keys() if isinstance(m, int) and m <= target_minute]
    
    if not relevant_minutes:
        return np.zeros(len(df))
        
    cols_to_sum = [col_map[m] for m in sorted(relevant_minutes)]
    return df[cols_to_sum].sum(axis=1)

# Generate dynamic time labels and calculate data
# We create new columns in the DF with a standard naming convention: prefix_0-Xmin
dynamic_time_labels = []
col_maps = {}

for prefix in METRIC_PREFIXES:
    col_maps[prefix] = get_time_columns(df, prefix)

for bin_min in CUSTOM_BINS_MINUTES:
    label = f"0-{bin_min}min"
    dynamic_time_labels.append(label)
    
    for prefix in METRIC_PREFIXES:
        new_col_name = f"{prefix}_{label}"
        df[new_col_name] = calculate_cumulative_bin(df, prefix, bin_min, col_maps[prefix])

# Also add 'pre' to labels if it exists in data
if 'pre' in col_maps['Di']: # Check one prefix, assume consistent
    dynamic_time_labels.insert(0, 'pre')

print(f"Available Dynamic Time Labels: {dynamic_time_labels}")

# ==========================================
# 3. HELPER FUNCTIONS
# ==========================================
def sem(x):
    x = np.asarray(x)
    return np.std(x, ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan

def format_pval(p):
    if pd.isna(p):
        return "p=NA"
    stars = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    if p >= 0.001:
        return f"{stars}\n(p={p:.3f})"
    else:
        return f"{stars}\n(p={p:.4f})"

def add_pval_annotation(ax, x1, x2, y_base, p_value, y_offset=0.12, height_level=0):
    y_top = y_base + y_offset + height_level * 0.08
    # ax.plot([x1, x1, x2, x2], [y_top, y_top + 0.04, y_top + 0.04, y_top], 
    #         color='black', linewidth=1.3, clip_on=False, zorder=4)
    label = format_pval(p_value)
    ax.text((x1 + x2) / 2, y_top + 0.06, label, 
            ha='center', va='bottom', fontsize=9, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='gray', alpha=0.9),
            zorder=5)

def calculate_pi(d_vals, s_vals):
    """Calculate Preference Index: PI = (D-S)/(D+S)"""
    pi_vals = []
    for d, s in zip(d_vals, s_vals):
        if (d + s) > 0:
            pi_vals.append((d - s) / (d + s))
        else:
            pi_vals.append(np.nan)
    return np.array(pi_vals)

# ==========================================
# 4. STATISTICAL TESTING: D vs S COMPARISONS
# ==========================================
print("=== Welch's t-tests: Chamber (D vs S) ===")
chamber_pvals = {}
for tp in dynamic_time_labels:
    for estr_val in [0, 1]:
        mask = df['Estrous'] == estr_val

        di_col = f"Di_{tp}" if tp != 'pre' else "Di_pre"
        do_col = f"Do_{tp}" if tp != 'pre' else "Do_pre"
        si_col = f"Si_{tp}" if tp != 'pre' else "Si_pre"
        so_col = f"So_{tp}" if tp != 'pre' else "So_pre"

        if di_col in df.columns and si_col in df.columns:
            D_val = df.loc[mask, di_col] + df.loc[mask, do_col] if do_col in df.columns else df.loc[mask, di_col]
            S_val = df.loc[mask, si_col] + df.loc[mask, so_col] if so_col in df.columns else df.loc[mask, si_col]
            
            if len(D_val)>1 and len(S_val)>1:
                t, p = stats.ttest_rel(D_val, S_val)
                key = (tp, 'Chamber', estr_val)
                chamber_pvals[key] = p
                estr_label = 'E' if estr_val==1 else 'Non-E'
                print(f"{tp:10s} {estr_label:6s} | t={t:6.3f} | p={p:.4f} {format_pval(p).split()[0]}")

print("\n=== Welch's t-tests: Sniffing (Di vs Si) ===")
inter_pvals = {}
for tp in dynamic_time_labels:
    for estr_val in [0, 1]:
        mask = df['Estrous'] == estr_val
        
        di_col = f"Di_{tp}" if tp != 'pre' else "Di_pre"
        si_col = f"Si_{tp}" if tp != 'pre' else "Si_pre"
        
        if di_col in df.columns and si_col in df.columns:
            D_val = df.loc[mask, di_col]
            S_val = df.loc[mask, si_col]
            
            if len(D_val)>1 and len(S_val)>1:
                t, p = stats.ttest_rel(D_val, S_val)
                key = (tp, 'Sniffing', estr_val)
                inter_pvals[key] = p
                estr_label = 'E' if estr_val==1 else 'Non-E'
                print(f"{tp:10s} {estr_label:6s} | t={t:6.3f} | p={p:.4f} {format_pval(p).split()[0]}")

# ==========================================
# PI STATISTICAL TESTING: ONE-SAMPLE T-TEST vs 0 (MODIFIED)
# ==========================================
print("\n=== One-sample t-tests: PI Index vs 0 ===")
pi_pvals_vs_zero = {}

for tp in dynamic_time_labels:
    for group_name in ['Chamber', 'Sniffing']:
        
        di_col = f"Di_{tp}" if tp != 'pre' else "Di_pre"
        do_col = f"Do_{tp}" if tp != 'pre' else "Do_pre"
        si_col = f"Si_{tp}" if tp != 'pre' else "Si_pre"
        so_col = f"So_{tp}" if tp != 'pre' else "So_pre"
        
        if di_col not in df.columns or si_col not in df.columns:
            continue

        for estr_val, estr_label in [(0, 'Non-E'), (1, 'E')]:
            mask = df['Estrous'] == estr_val
            
            if group_name == 'Chamber':
                D_data = df.loc[mask, di_col] + (df.loc[mask, do_col] if do_col in df.columns else 0)
                S_data = df.loc[mask, si_col] + (df.loc[mask, so_col] if so_col in df.columns else 0)
            else:
                D_data = df.loc[mask, di_col]
                S_data = df.loc[mask, si_col]
            
            pi_vals = calculate_pi(D_data, S_data)
            pi_vals_clean = pi_vals[~np.isnan(pi_vals)]
            
            if len(pi_vals_clean) > 1:
                # ONE-SAMPLE T-TEST AGAINST 0
                t_stat, p_val = stats.ttest_1samp(pi_vals_clean, popmean=0)
                key = (tp, group_name, estr_val)
                pi_pvals_vs_zero[key] = p_val
                mean_pi = np.mean(pi_vals_clean)
                direction = "D-pref" if mean_pi > 0 else "S-pref" if mean_pi < 0 else "neutral"
                print(f"{tp:10s} {group_name:10s} {estr_label:6s} | Mean PI={mean_pi:.3f} ({direction}) | t={t_stat:6.3f} | p={p_val:.4f} {format_pval(p_val).split()[0]}")
            else:
                key = (tp, group_name, estr_val)
                pi_pvals_vs_zero[key] = np.nan

# ==========================================
# 5. PLOT 1 (MODIFIED): 2 ROWS × N COLUMNS - ESTRUS TOP, NON-E BOTTOM
# ==========================================
fig1, axes1 = plt.subplots(2, len(dynamic_time_labels), figsize=(6*len(dynamic_time_labels), 10), sharey=True)
if len(dynamic_time_labels) == 1: 
    axes1 = np.array([[axes1[0]], [axes1[1]]])  # Ensure 2D indexing

colors = {'D': '#2E86AB', 'S': '#E94F37'}
hatches = {'E': '///', 'NonE': ''}
estr_order = [(1, 'Estrous'), (0, 'Non-Estrous')]  # Top row = Estrous

for row_idx, (estr_val, estr_title) in enumerate(estr_order):
    for col_idx, tp in enumerate(dynamic_time_labels):
        ax = axes1[row_idx, col_idx]
        
        group_positions = [0, 2]
        bar_width = 0.8
        group_labels = ['Chamber', 'Sniffing']
        
        for grp_idx, group_name in enumerate(['Chamber', 'Sniffing']):
            base_pos = group_positions[grp_idx]
            
            di_col = f"Di_{tp}" if tp != 'pre' else "Di_pre"
            do_col = f"Do_{tp}" if tp != 'pre' else "Do_pre"
            si_col = f"Si_{tp}" if tp != 'pre' else "Si_pre"
            so_col = f"So_{tp}" if tp != 'pre' else "So_pre"
            
            mask = df['Estrous'] == estr_val
            
            if group_name == 'Chamber':
                D_data = df.loc[mask, di_col] + (df.loc[mask, do_col] if do_col in df.columns else 0)
                S_data = df.loc[mask, si_col] + (df.loc[mask, so_col] if so_col in df.columns else 0)
                subset = df[mask].copy()
                subset['D_val'] = subset[di_col] + (subset[do_col] if do_col in df.columns else 0)
                subset['S_val'] = subset[si_col] + (subset[so_col] if so_col in df.columns else 0)
                paired_data = subset[['Fem', 'D_val', 'S_val']].dropna()
                p_key = (tp, 'Chamber', estr_val)
                p_val = chamber_pvals.get(p_key)
            else:
                D_data = df.loc[mask, di_col]
                S_data = df.loc[mask, si_col]
                subset = df[mask].copy()
                subset['D_val'] = subset[di_col]
                subset['S_val'] = subset[si_col]
                paired_data = subset[['Fem', 'D_val', 'S_val']].dropna()
                p_key = (tp, 'Sniffing', estr_val)
                p_val = inter_pvals.get(p_key)
            
            estr_key = 'E' if estr_val==1 else 'NonE'
            
            # D bar
            if len(D_data) > 0:
                mean_D = D_data.mean()
                sem_D = sem(D_data)
                ax.bar(base_pos, mean_D, width=bar_width, 
                       color=colors['D'], hatch=hatches[estr_key], edgecolor='white', 
                       linewidth=1.2, alpha=0.5,
                       label=f'D' if row_idx==0 and col_idx==0 and grp_idx==0 else "",
                       zorder=2)
                ax.errorbar(base_pos, mean_D, yerr=sem_D, fmt='none', 
                            ecolor='black', capsize=4, capthick=1.5, linewidth=1.5, zorder=3)
                jitter = np.random.normal(0, 0.03, size=len(D_data))
                ax.scatter(base_pos + jitter, D_data, c='black', s=20,
                           alpha=0.8, edgecolors='white', linewidth=0.5, zorder=4)
            
            # S bar
            if len(S_data) > 0:
                mean_S = S_data.mean()
                sem_S = sem(S_data)
                ax.bar(base_pos + bar_width, mean_S, width=bar_width, 
                       color=colors['S'], hatch=hatches[estr_key], edgecolor='white',
                       linewidth=1.2, alpha=0.5,
                       label=f'S' if row_idx==0 and col_idx==0 and grp_idx==0 else "",
                       zorder=2)
                ax.errorbar(base_pos + bar_width, mean_S, yerr=sem_S, fmt='none', 
                            ecolor='black', capsize=4, capthick=1.5, linewidth=1.5, zorder=3)
                jitter = np.random.normal(0, 0.03, size=len(S_data))
                ax.scatter(base_pos + bar_width + jitter, S_data, c='black', s=20,
                           alpha=0.8, edgecolors='white', linewidth=0.5, zorder=4)
            
            # Connect paired D/S dots
            for _, row in paired_data.iterrows():
                d_val, s_val = row['D_val'], row['S_val']
                ax.plot([base_pos + np.random.normal(0, 0.02), base_pos + bar_width + np.random.normal(0, 0.02)], 
                        [d_val, s_val], color='darkgray', linewidth=1, alpha=0.7, zorder=3)
            
            # P-value annotation
            if p_val is not None and not np.isnan(p_val):
                y_max = max((mean_D + sem_D if len(D_data)>0 else 0), (mean_S + sem_S if len(S_data)>0 else 0))
                add_pval_annotation(ax, base_pos, base_pos + bar_width, y_max, p_val, y_offset=0.2)
        
        ax.set_title(f'{tp}', fontsize=18, fontweight='bold', pad=8)
        ax.set_xticks([pos + bar_width/2 for pos in group_positions])
        ax.set_xticklabels(group_labels, fontsize=14, ha='center')
        ax.grid(axis='y', alpha=0.35, linestyle='--', linewidth=0.5)
        ax.axhline(0, color='gray', linewidth=0.5, linestyle=':', zorder=1)
        ax.tick_params(axis='x', length=0)
        
        # Only add y-label to leftmost column
        if col_idx == 0: 
            ax.set_ylabel('Time [seconds]', fontsize=14, labelpad=5)
    
    # Add row label on the far left
    axes1[row_idx, 0].text(-0.15, 0.5, estr_title, rotation=90, va='center', ha='center', 
                          fontsize=15, fontweight='bold', transform=axes1[row_idx, 0].transAxes,
                          bbox=dict(boxstyle='round,pad=0.4', facecolor='lightgray', alpha=0.3))

# Legend
legend_elements = [
    Patch(facecolor='#2E86AB', edgecolor='white', alpha=0.5, label='D time'),
    Patch(facecolor='#E94F37', edgecolor='white', alpha=0.5, label='S time'),
    Patch(facecolor='#2E86AB', edgecolor='white', hatch='///', alpha=0.5, label='Estrous'),
    Patch(facecolor='#2E86AB', edgecolor='white', alpha=0.5, label='Non-Estrous'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='black', markersize=5, alpha=0.8, linestyle='None', label='Raw data'),
    plt.Line2D([0], [0], color='darkgray', linewidth=1, alpha=0.7, label='Paired subjects')
]
fig1.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, 0.01), ncol=6, frameon=True, fontsize=10)

plt.suptitle('Time: D vs S Comparison by Estrous Status (Mean ± SEM)', fontsize=13, fontweight='bold', y=1.00)
plt.tight_layout(rect=[0, 0.02, 1, 0.96])
plt.show()

# ==========================================
# 6. PLOT 3 (MODIFIED): PI INDEX - 2 ROWS × N COLUMNS
# Columns: Time points | Each subplot: Chamber PI + Sniffing PI
# ==========================================
fig3, axes3 = plt.subplots(2, len(dynamic_time_labels), figsize=(6*len(dynamic_time_labels), 9), sharey=True)
if len(dynamic_time_labels) == 1: 
    axes3 = np.array([[axes3[0]], [axes3[1]]])  # Ensure 2D indexing

pi_colors = {'positive': "#2E86AB", 'negative': '#E94F37'}  # Blue for D-pref, Red for S-pref
pi_hatches = {'E': '///', 'NonE': ''}
estr_order = [(1, 'Estrous'), (0, 'Non-Estrous')]  # Top row = Estrous

for row_idx, (estr_val, estr_title) in enumerate(estr_order):
    for col_idx, tp in enumerate(dynamic_time_labels):
        ax = axes3[row_idx, col_idx]
        
        group_positions = [0, 1]  # Chamber at 0, Sniffing at 3
        bar_width = 0.8
        group_labels = ['Chamber', 'Sniffing']
        
        for grp_idx, group_name in enumerate(['Chamber', 'Sniffing']):
            base_pos = group_positions[grp_idx]
            
            mask = df['Estrous'] == estr_val
            
            di_col = f"Di_{tp}" if tp != 'pre' else "Di_pre"
            do_col = f"Do_{tp}" if tp != 'pre' else "Do_pre"
            si_col = f"Si_{tp}" if tp != 'pre' else "Si_pre"
            so_col = f"So_{tp}" if tp != 'pre' else "So_pre"
            
            if di_col not in df.columns or si_col not in df.columns:
                continue

            # Calculate D and S values based on group type
            if group_name == 'Chamber':
                D_data = df.loc[mask, di_col] + (df.loc[mask, do_col] if do_col in df.columns else 0)
                S_data = df.loc[mask, si_col] + (df.loc[mask, so_col] if so_col in df.columns else 0)
            else:  # Sniffing
                D_data = df.loc[mask, di_col]
                S_data = df.loc[mask, si_col]
            
            # Calculate Preference Index
            pi_vals = calculate_pi(D_data, S_data)
            pi_vals_clean = pi_vals[~np.isnan(pi_vals)]
            
            estr_key = 'E' if estr_val==1 else 'NonE'
            
            if len(pi_vals_clean) > 0:
                mean_pi = np.mean(pi_vals_clean)
                sem_pi = sem(pi_vals_clean)
                
                # Color bar based on direction of preference
                bar_color = pi_colors['positive'] if mean_pi >= 0 else pi_colors['negative']
                
                # Plot bar
                bar_center = base_pos + bar_width/2
                ax.bar(bar_center, mean_pi, width=bar_width, 
                       color=bar_color, hatch=pi_hatches[estr_key], 
                       edgecolor='white', linewidth=1.2, alpha=0.6,
                       label=f'{group_name}' if row_idx==0 and col_idx==0 and grp_idx==0 else "",
                       zorder=2)
                
                # Error bar
                ax.errorbar(bar_center, mean_pi, yerr=sem_pi, fmt='none', 
                            ecolor='black', capsize=4, capthick=1.5, linewidth=1.5, zorder=3)
                
                # Scatter individual PI values with jitter
                jitter = np.random.normal(0, 0.05, size=len(pi_vals_clean))
                ax.scatter(bar_center + jitter, pi_vals_clean, 
                           c='black', s=25, alpha=0.7, edgecolors='white', 
                           linewidth=0.5, zorder=4)
                
                # Statistical annotation: one-sample t-test vs 0
                p_key = (tp, group_name, estr_val)
                p_val = pi_pvals_vs_zero.get(p_key)
                
                if p_val is not None and not np.isnan(p_val):
                    # Draw vertical line from bar top to annotation
                    y_base = abs(mean_pi) + 0.08
                    y_top = y_base + 0.04
                    label = format_pval(p_val)
                    ax.text(bar_center, y_top + 0.02, label, 
                            ha='center', va='bottom', fontsize=9, fontweight='bold',
                            bbox=dict(boxstyle='round,pad=0.25', facecolor='white', edgecolor='gray', alpha=0.95),
                            zorder=6)
        
        # Subplot formatting
        ax.set_title(f'{tp}', fontsize=18, fontweight='bold', pad=8)
        ax.set_xticks([pos + bar_width/2 for pos in group_positions])
        ax.set_xticklabels(group_labels, fontsize=14, ha='center')
        ax.grid(axis='y', alpha=0.35, linestyle='--', linewidth=0.5)
        ax.axhline(0, color='gray', linewidth=1, linestyle='-', zorder=1)  # Zero reference line
        ax.set_ylim(-0.7, 0.7)  # Fixed y-range for PI
        ax.tick_params(axis='x', length=0)
        
        # Y-label only on leftmost column
        if col_idx == 0: 
            ax.set_ylabel('Preference Index', fontsize=14, labelpad=5)
    
    # Add row label on far left margin
    axes3[row_idx, 0].text(-0.18, 0.5, estr_title, rotation=90, va='center', ha='center', 
                          fontsize=14, fontweight='bold', transform=axes3[row_idx, 0].transAxes,
                          bbox=dict(boxstyle='round,pad=0.4', facecolor='lightgray', alpha=0.35))

# Legend for PI plot
pi_legend_elements = [
    Patch(facecolor='#2E86AB', edgecolor='white', alpha=0.6, label='PI > 0 (D preference)'),
    Patch(facecolor='#E94F37', edgecolor='white', alpha=0.6, label='PI < 0 (S preference)'),
    Patch(facecolor='#2E86AB', edgecolor='white', hatch='///', alpha=0.6, label='Estrous'),
    Patch(facecolor='#2E86AB', edgecolor='white', alpha=0.6, label='Non-Estrous'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='black', 
               markersize=5, alpha=0.7, linestyle='None', label='Individual PI'),
    plt.Line2D([0], [0], color='black', linewidth=1.5, label='± SEM'),
    plt.Line2D([0], [0], color='black', linewidth=0.8, label='vs. 0 (one-sample t-test)')
]
fig3.legend(handles=pi_legend_elements, loc='upper center', bbox_to_anchor=(0.5, 0.01), 
            ncol=7, frameon=True, fontsize=9, columnspacing=1.2)

plt.suptitle('Preference Index (PI): One-Sample t-Test vs Zero by Estrous Status\n'
             'PI = (D−S)/(D+S) | Annotations: *p<0.05, **p<0.01, ***p<0.001', 
             fontsize=13, fontweight='bold', y=1.00)
plt.tight_layout(rect=[0, 0.02, 1, 0.96])
plt.show()

# ==========================================
# 7. PLOT 2: TIME COURSE - D AND S LINES SIDE-BY-SIDE (CUMULATIVE)
# ==========================================
# Use dynamic_time_labels for the time course as well
time_points_tc = dynamic_time_labels

fig2, axes2 = plt.subplots(2, 2, figsize=(14, 10))

for row_idx, (metric_name, d_col1, d_col2, s_col1, s_col2, ylabel) in enumerate([
    ('Chamber', 'Di', 'Do', 'Si', 'So', 'Chamber Time [seconds]'),
    ('Sniffing', 'Di', None, 'Si', None, 'Sniffing Time [seconds]')
]):
    for col_idx, (estr_val, estr_title) in enumerate([(0, 'Non-Estrous'), (1, 'Estrous')]):
        ax = axes2[row_idx, col_idx]
        
        plot_data = []
        for _, row in df.iterrows():
            if row['Estrous'] == estr_val:
                fem = row['Fem']
                for tp in time_points_tc:
                    # Construct column names dynamically
                    d_c1 = f"{d_col1}_{tp}" if tp != 'pre' else f"{d_col1}_pre"
                    s_c1 = f"{s_col1}_{tp}" if tp != 'pre' else f"{s_col1}_pre"
                    
                    d_val = row[d_c1] if d_c1 in df.columns else 0
                    if d_col2:
                        d_c2 = f"{d_col2}_{tp}" if tp != 'pre' else f"{d_col2}_pre"
                        d_val += row[d_c2] if d_c2 in df.columns else 0
                        
                    s_val = row[s_c1] if s_c1 in df.columns else 0
                    if s_col2:
                        s_c2 = f"{s_col2}_{tp}" if tp != 'pre' else f"{s_col2}_pre"
                        s_val += row[s_c2] if s_c2 in df.columns else 0
                        
                    plot_data.append({'Fem': fem, 'Time': tp, 'D': d_val, 'S': s_val})
        
        tc_df = pd.DataFrame(plot_data)
        tc_df['Time'] = pd.Categorical(tc_df['Time'], categories=time_points_tc, ordered=True)
        
        x_pos = np.arange(len(time_points_tc))
        
        d_means = [tc_df[tc_df['Time']==tp]['D'].mean() for tp in time_points_tc]
        d_sems = [sem(tc_df[tc_df['Time']==tp]['D']) for tp in time_points_tc]
        ax.plot(x_pos, d_means, color='#2E86AB', linewidth=2.5, marker='o', 
                markersize=6, label='D time', zorder=3)
        ax.fill_between(x_pos, [m-s for m,s in zip(d_means, d_sems)], 
                        [m+s for m,s in zip(d_means, d_sems)], 
                        color='#2E86AB', alpha=0.2, zorder=2)
        
        s_means = [tc_df[tc_df['Time']==tp]['S'].mean() for tp in time_points_tc]
        s_sems = [sem(tc_df[tc_df['Time']==tp]['S']) for tp in time_points_tc]
        ax.plot(x_pos, s_means, color='#E94F37', linewidth=2.5, marker='s', 
                markersize=6, label='S time', zorder=3)
        ax.fill_between(x_pos, [m-s for m,s in zip(s_means, s_sems)], 
                        [m+s for m,s in zip(s_means, s_sems)], 
                        color='#E94F37', alpha=0.2, zorder=2)
        
        fems_complete = tc_df.groupby('Fem').filter(lambda g: len(g)==len(time_points_tc))['Fem'].unique()
        for fem in fems_complete:
            fem_data = tc_df[tc_df['Fem']==fem].sort_values('Time')
            d_vals = fem_data['D'].values
            if not np.any(np.isnan(d_vals)):
                ax.plot(x_pos, d_vals, color='#2E86AB', linewidth=0.6, alpha=0.3, zorder=1)
            s_vals = fem_data['S'].values
            if not np.any(np.isnan(s_vals)):
                ax.plot(x_pos, s_vals, color='#E94F37', linewidth=0.6, alpha=0.3, zorder=1)
        
        for i, tp in enumerate(time_points_tc):
            D_vals = tc_df[tc_df['Time']==tp]['D'].dropna()
            S_vals = tc_df[tc_df['Time']==tp]['S'].dropna()
            
            if len(D_vals) > 1 and len(S_vals) > 1 and len(D_vals) == len(S_vals):
                t_stat, p_val = stats.ttest_rel(D_vals, S_vals)
                
                y_max = max(d_means[i] + d_sems[i], s_means[i] + s_sems[i])
                add_pval_annotation(ax, i-0.15, i+0.15, y_max, p_val, y_offset=100.0)
        

        ax.set_title(f'{estr_title} (n={len(fems_complete)})', fontsize=18, fontweight='bold', pad=10)
        ax.set_xticks(x_pos)

        clean_labels = [tp.replace('0-','').replace('min',' min') if 'min' in tp else tp for tp in time_points_tc]
        ax.set_xticklabels(clean_labels, fontsize=9, rotation=45, ha='right')
        ax.grid(axis='y', alpha=0.35, linestyle='--', linewidth=0.5)

        if col_idx == 0: ax.set_ylabel(ylabel, fontsize=14, labelpad=8)
        if row_idx == 1: ax.set_xlabel('Time window', fontsize=14)
        ax.tick_params(axis='x', length=0)
        if row_idx == 0 and col_idx == 1:
            ax.legend(fontsize=9, frameon=True)

plt.suptitle('Actual Time Course (Cumulative): D vs S Trajectories\n'
             'Solid lines: group mean ± SEM; Faint lines: individual subjects',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(rect=[0, 0.03, 1, 0.94])
plt.show()

# ==========================================
# 8. PLOT 4: NON-CUMULATIVE TIME COURSE + PAIRED T-TESTS (RAW P-VALUES)
# ==========================================

fig4, axes4 = plt.subplots(2, 2, figsize=(14, 10), sharex=True)

# Prepare discrete time bins
cumulative_cols = [col for col in dynamic_time_labels if col.startswith('0-')]
def get_min_from_label(label):
    return int(label.split('-')[1].replace('min', ''))
cumulative_cols.sort(key=get_min_from_label)

discrete_map = []
for i, cum_col in enumerate(cumulative_cols):
    current_min = get_min_from_label(cum_col)
    if i == 0:
        prev_min, prev_col, label = 0, None, f"0-{current_min} min"
    else:
        prev_min = get_min_from_label(cumulative_cols[i-1])
        prev_col = cumulative_cols[i-1]
        label = f"{prev_min}-{current_min} min"
    discrete_map.append({'label': label, 'curr_col': cum_col, 'prev_col': prev_col, 'minute': current_min})

ordered_labels = [item['label'] for item in discrete_map]

# Helper: calculate discrete values for a time bin
def get_discrete_value(row, prefix, curr_col, prev_col, alt_prefix=None):
    def get_val(p, c):
        full = f"{p}_{c}" if c else None
        return row[full] if full and full in df.columns else 0
    curr = get_val(prefix, curr_col)
    if alt_prefix: curr += get_val(alt_prefix, curr_col)
    prev = get_val(prefix, prev_col) if prev_col else 0
    if alt_prefix and prev_col: prev += get_val(alt_prefix, prev_col)
    return max(0, curr - prev)

for row_idx, (metric_name, d_col1, d_col2, s_col1, s_col2, ylabel) in enumerate([
    ('Chamber', 'Di', 'Do', 'Si', 'So', 'Chamber Time [seconds]'),
    ('Sniffing', 'Di', None, 'Si', None, 'Sniffing Time [seconds]')
]):
    for col_idx, (estr_val, estr_title) in enumerate([(0, 'Non-Estrous'), (1, 'Estrous')]):
        ax = axes4[row_idx, col_idx]
        mask = df['Estrous'] == estr_val
        df_subset = df[mask]
        
        # 🔹 Prepare plot data AND collect p-values for annotation
        plot_data = []
        pval_by_idx = {}  # Store raw p-values for annotation
        
        for item_idx, item in enumerate(discrete_map):
            D_vals, S_vals = [], []
            for _, row in df_subset.iterrows():
                d_val = get_discrete_value(row, d_col1, item['curr_col'], item['prev_col'], d_col2)
                s_val = get_discrete_value(row, s_col1, item['curr_col'], item['prev_col'], s_col2)
                D_vals.append(d_val)
                S_vals.append(s_val)
            
            D_arr, S_arr = np.array(D_vals), np.array(S_vals)
            
            # 🔹 Paired t-test (RAW p-value, NO FDR correction)
            if len(D_arr) > 1 and len(S_arr) > 1:
                t_stat, p_val = stats.ttest_rel(D_arr, S_arr)
                pval_by_idx[item_idx] = p_val  # Store for annotation
            else:
                pval_by_idx[item_idx] = np.nan
            
            # Add to plot data
            for fem_id, d_val, s_val in zip(df_subset['Fem'], D_arr, S_arr):
                plot_data.append({'Fem': fem_id, 'Time': item['label'], 'D': d_val, 'S': s_val})
        
        tc_df = pd.DataFrame(plot_data)
        tc_df['Time'] = pd.Categorical(tc_df['Time'], categories=ordered_labels, ordered=True)
        x_pos = np.arange(len(ordered_labels))
        
        # Plot means ± SEM
        d_means = [tc_df[tc_df['Time']==tp]['D'].mean() for tp in ordered_labels]
        d_sems = [sem(tc_df[tc_df['Time']==tp]['D']) for tp in ordered_labels]
        s_means = [tc_df[tc_df['Time']==tp]['S'].mean() for tp in ordered_labels]
        s_sems = [sem(tc_df[tc_df['Time']==tp]['S']) for tp in ordered_labels]
        
        ax.plot(x_pos, d_means, color='#2E86AB', linewidth=2.5, marker='o', markersize=6, label='D time', zorder=3)
        ax.fill_between(x_pos, [m-s for m,s in zip(d_means, d_sems)], [m+s for m,s in zip(d_means, d_sems)], color='#2E86AB', alpha=0.2, zorder=2)
        ax.plot(x_pos, s_means, color='#E94F37', linewidth=2.5, marker='s', markersize=6, label='S time', zorder=3)
        ax.fill_between(x_pos, [m-s for m,s in zip(s_means, s_sems)], [m+s for m,s in zip(s_means, s_sems)], color='#E94F37', alpha=0.2, zorder=2)
        
        # Individual lines
        fems_complete = tc_df.groupby('Fem').filter(lambda g: len(g)==len(ordered_labels))['Fem'].unique()
        for fem in fems_complete:
            fem_data = tc_df[tc_df['Fem']==fem].sort_values('Time')
            if not fem_data['D'].isna().any():
                ax.plot(x_pos, fem_data['D'], color='#2E86AB', linewidth=0.6, alpha=0.3, zorder=1)
            if not fem_data['S'].isna().any():
                ax.plot(x_pos, fem_data['S'], color='#E94F37', linewidth=0.6, alpha=0.3, zorder=1)
        
        # 🔹 Annotate with RAW p-values using common helper (no FDR)
        for i, item in enumerate(discrete_map):
            p_val = pval_by_idx.get(i)
            if p_val is not None and not np.isnan(p_val):
                y_max = max(d_means[i] + d_sems[i], s_means[i] + s_sems[i])
                # Use bracket-style annotation centered on time point
                add_pval_annotation(ax, i-0.2, i+0.2, y_max, p_val, y_offset=100.0)
        
        # Formatting
        ax.set_title(f'{estr_title} (n={len(fems_complete)})', fontsize=18, fontweight='bold', pad=10)
        ax.set_xticks(x_pos)
        ax.set_xticklabels(ordered_labels, fontsize=14, rotation=45, ha='right')
        ax.grid(axis='y', alpha=0.35, linestyle='--', linewidth=0.5)
        if col_idx == 0: ax.set_ylabel(ylabel, fontsize=14, labelpad=8)
        if row_idx == 1: ax.set_xlabel('Time Interval (Minutes)', fontsize=14)
        ax.tick_params(axis='x', length=0)
        if row_idx == 0 and col_idx == 1:
            ax.legend(fontsize=9, frameon=True)

plt.suptitle('Non-Cumulative Time Course: Paired T-Tests (Raw P-Values)\n'
             'Values: Time in specific bin only | Annotations: paired t-test D vs S (*p<0.05, **p<0.01, ***p<0.001, ns=not significant)', 
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(rect=[0, 0.03, 1, 0.94])
plt.show()

# ==========================================
# 9. PLOT 5 (FIXED): 20-30 MIN DISCRETE INTERVAL
# ==========================================
fig5, axes5 = plt.subplots(1, 2, figsize=(12, 6), sharey=True)

# 🔹 Calculate DISCRETE 20-30 min values: cumulative_30min - cumulative_20min
def get_discrete_20_30(df_subset, prefix, alt_prefix=None):
    """Calculate discrete time spent in 20-30 min window from a pre-filtered subset"""
    col_30 = f"{prefix}_0-30min"
    col_20 = f"{prefix}_0-20min"
    
    # Return zeros if columns don't exist
    if col_30 not in df_subset.columns or col_20 not in df_subset.columns:
        return np.zeros(len(df_subset))
    
    val_30 = df_subset[col_30].fillna(0).values
    val_20 = df_subset[col_20].fillna(0).values
    
    if alt_prefix:
        alt_30_col = f"{alt_prefix}_0-30min"
        alt_20_col = f"{alt_prefix}_0-20min"
        if alt_30_col in df_subset.columns:
            val_30 += df_subset[alt_30_col].fillna(0).values
        if alt_20_col in df_subset.columns:
            val_20 += df_subset[alt_20_col].fillna(0).values
    
    return np.maximum(0, val_30 - val_20)

metrics = [
    {'name': 'Chamber Time', 'd_prefix': 'Di', 'd_alt': 'Do', 's_prefix': 'Si', 's_alt': 'So'},
    {'name': 'Sniffing Time', 'd_prefix': 'Di', 'd_alt': None, 's_prefix': 'Si', 's_alt': None}
]

for idx, metric in enumerate(metrics):
    ax = axes5[idx]
    plot_groups = []
    
    for estr_val, estr_label in [(0, 'Non-Estrous'), (1, 'Estrous')]:
        # Filter subset first
        mask = df['Estrous'] == estr_val
        subset = df[mask].copy()  # .copy() to avoid SettingWithCopyWarning
        
        # 🔹 Get discrete 20-30 min values (already filtered to subset)
        D_vals = get_discrete_20_30(subset, metric['d_prefix'], metric['d_alt'])
        S_vals = get_discrete_20_30(subset, metric['s_prefix'], metric['s_alt'])

        plot_groups.append({
            'label': estr_label, 
            'estr_val': estr_val,
            'D': pd.Series(D_vals, index=subset.index),  # Use subset.index for alignment
            'S': pd.Series(S_vals, index=subset.index)
        })

    # Plotting parameters
    x_positions = [0, 2]
    bar_width = 0.8
    
    for i, grp in enumerate(plot_groups):
        base_pos = x_positions[i]
        
        # Clean data and ensure pairing
        D_clean = grp['D'].dropna()
        S_clean = grp['S'].dropna()
        
        # Align by index to ensure true pairing
        common_idx = D_clean.index.intersection(S_clean.index)
        D_paired = D_clean.loc[common_idx]
        S_paired = S_clean.loc[common_idx]
        
        # D Bar
        if len(D_paired) > 0:
            mean_D, sem_D = D_paired.mean(), sem(D_paired)
            ax.bar(base_pos, mean_D, width=bar_width, color='#2E86AB', 
                   hatch='///' if grp['estr_val']==1 else '', edgecolor='white', alpha=0.6, zorder=2)
            ax.errorbar(base_pos, mean_D, yerr=sem_D, fmt='none', ecolor='black', capsize=4, zorder=3)
            jitter = np.random.normal(0, 0.02, size=len(D_paired))
            ax.scatter(base_pos + jitter, D_paired, c='black', s=25, alpha=0.8, edgecolors='white', zorder=4)
        
        # S Bar
        if len(S_paired) > 0:
            mean_S, sem_S = S_paired.mean(), sem(S_paired)
            ax.bar(base_pos + bar_width, mean_S, width=bar_width, color='#E94F37',
                   hatch='///' if grp['estr_val']==1 else '', edgecolor='white', alpha=0.6, zorder=2)
            ax.errorbar(base_pos + bar_width, mean_S, yerr=sem_S, fmt='none', ecolor='black', capsize=4, zorder=3)
            jitter = np.random.normal(0, 0.02, size=len(S_paired))
            ax.scatter(base_pos + bar_width + jitter, S_paired, c='black', s=25, alpha=0.8, edgecolors='white', zorder=4)
        
        # Paired lines (only for truly matched pairs)
        if len(D_paired) == len(S_paired) and len(D_paired) > 0:
            for d, s in zip(D_paired, S_paired):
                ax.plot([base_pos + np.random.normal(0,0.015), base_pos + bar_width + np.random.normal(0,0.015)], 
                        [d, s], color='gray', linewidth=0.8, alpha=0.6, zorder=1)
        
        # 🔹 Statistical test: paired t-test within group (only on aligned pairs)
        if len(D_paired) > 1 and len(S_paired) > 1:
            t_stat, p_val = stats.ttest_rel(D_paired, S_paired)
            stars = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else None
            if stars:
                y_max = max(mean_D if len(D_paired)>0 else 0, mean_S if len(S_paired)>0 else 0) + 20
                add_pval_annotation(ax, base_pos, base_pos + bar_width, y_max, p_val, y_offset=0.25)

    ax.set_title(f"{metric['name']}\n(20-30 min Discrete)", fontsize=12, fontweight='bold', pad=10)
    ax.set_xticks([pos + bar_width/2 for pos in x_positions])
    ax.set_xticklabels(['Non-Estrous', 'Estrous'], fontsize=14)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    if idx == 0:
        ax.set_ylabel('Time [seconds]', fontsize=14, labelpad=5)
    ax.tick_params(axis='x', length=0)

# Legend
legend_elements = [
    Patch(facecolor='#2E86AB', edgecolor='white', alpha=0.6, label='D time'),
    Patch(facecolor='#E94F37', edgecolor='white', alpha=0.6, label='S time'),
    Patch(facecolor='#2E86AB', edgecolor='white', hatch='///', alpha=0.6, label='Estrous'),
    Patch(facecolor='#2E86AB', edgecolor='white', alpha=0.6, label='Non-Estrous'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='black', markersize=4, alpha=0.8, linestyle='None', label='Individual'),
    plt.Line2D([0], [0], color='gray', linewidth=0.8, alpha=0.6, label='Paired subjects')
]
fig5.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, 0.01), ncol=6, frameon=True, fontsize=9)

plt.suptitle('Behavior in 20-30 Minute Window (Discrete, Non-Cumulative)\n'
             'Values = Cumulative(0-30min) − Cumulative(0-20min) | Paired t-test within estrous group', 
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(rect=[0, 0.03, 1, 0.96])
plt.show()